In [143]:
## Import libraries (numpy, pandas, ...)
import pandas as pd
import numpy as np
from typing import Dict
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import requests


In [144]:
## Import Divvy trip data
divvy_trips = pd.read_stata("data/divvy_data.dta")
divvy_trips

,start_date,from_station_id,trips
0,2013-06-27,17,4.0
1,2013-06-27,19,2.0
2,2013-06-27,20,1.0
3,2013-06-27,24,1.0
4,2013-06-27,28,1.0
...,...,...,...
951667,2019-12-31,659,3.0
951668,2019-12-31,660,2.0
951669,2019-12-31,664,1.0
951670,2019-12-31,672,21.0


In [145]:
## Import Divvy location dataset
divvy_locations = pd.read_stata("data/IDlatlong.dta")
divvy_locations

,from_station_id,Latitude,Longitude
0,2.0,41.876511,-87.620548
1,3.0,41.867226,-87.615355
2,4.0,41.856268,-87.613348
3,5.0,41.874053,-87.627716
4,6.0,41.886976,-87.612813
...,...,...,...
604,664.0,41.939354,-87.683282
605,665.0,41.747363,-87.580046
606,666.0,41.907221,-87.655618
607,672.0,41.891023,-87.635480


In [146]:
## Merge Datasets
divvy = pd.merge(divvy_trips, divvy_locations, left_on="from_station_id", right_on="from_station_id", how="left")
divvy

,start_date,from_station_id,trips,Latitude,Longitude
0,2013-06-27,17,4.0,41.903119,-87.673935
1,2013-06-27,19,2.0,41.868968,-87.659141
2,2013-06-27,20,1.0,41.910522,-87.653106
3,2013-06-27,24,1.0,41.891847,-87.620580
4,2013-06-27,28,1.0,41.914680,-87.643320
...,...,...,...,...,...
951667,2019-12-31,659,3.0,41.895501,-87.682017
951668,2019-12-31,660,2.0,42.004583,-87.661406
951669,2019-12-31,664,1.0,41.939354,-87.683282
951670,2019-12-31,672,21.0,41.891023,-87.635480


In [147]:
# Haversine distance in miles
def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c



def div_filter( divvy : pd.DataFrame, c : tuple[float], r :float):
    '''
    dataset filtering function; takes center, radius info for filtering
    params: 
        divvy (DataFrame): DataFrame w/ divvy bike ride info including longitude and latitude values for station, station ID, and date (YYY-MM-DD)
        c (tuple(float)) : (c is short for "center"), this is the center longitude and latitude tuple defining the midpoint of the area for which the divvy data will be filtered for distance 
        r (float) : radius (in Mi), the radius defining the circular boundary around the center ('c') point (divvy ride entried within this radius will be added to the output dataframe)
    outputs: 
        inrange (DataFrame) : a dataframe filtered for divvy bikle rides within the specified radius around the provided center point
    '''

    # coordinates lat, long sep into single vars for computations
    lat = c[0]
    lon = c[1]

    # Calculate distance from each station to Soldier Field
    divvy["dist_to_c_mi"] = haversine_miles(
        lat,
        lon,
        divvy["Latitude"],
        divvy["Longitude"]
    )

    inrange = divvy[divvy["dist_to_c_mi"] <= r].copy()
    return inrange
  

In [148]:
def split_df_on_game_days(divvy: pd.DataFrame, gds_by_yr: dict):
    all_gds = set()
    for dates in gds_by_yr.values():
        all_gds |= dates

    # Build a mask: within any season's first-to-last game range
    in_season = pd.Series(False, index=divvy.index)
    for yr_dates in gds_by_yr.values():
        season_start = min(yr_dates)
        season_end   = max(yr_dates)
        in_season |= (divvy["start_date"] >= season_start) & (divvy["start_date"] <= season_end)

    game_day_mask = divvy["start_date"].isin(all_gds)

    gd    = divvy[game_day_mask].copy(); print("GAME DAY"); print(gd)
    notgd = divvy[in_season & ~game_day_mask].copy(); print("\033[1m" +"*NOT* " + "\033[0m" + "GAME DAY"); print(notgd)

    return gd, notgd


In [149]:
#make coords for all tmt and ctrl areas and compile respective datasets for ea.
soldiers_coords = (41.8625, -87.6167)
# returns df of all divvy ride entries within 1 mi of soldiers field center pt as defined above
soldiers_df = div_filter(divvy, soldiers_coords, r=1.0) 

jackson_coords = (41.7831, -87.5819)
jackson_df = div_filter(divvy, jackson_coords, r=1.0) # does the same but now for jackson park (control area)

wicker_coords = (41.907744, -87.67676)
wicker_df = div_filter(divvy, wicker_coords, r=1.0) # does the same but now for wicker park (alternate option for control area)


#now need to create a dict to hold the game days by yr, will use the years as keys then use a set to hold the dates in same format as the divvy df does ('YYY-MM-DD' string)
gds_by_yr = {
    '2017': {
        pd.Timestamp('2017-09-10'),
        pd.Timestamp('2017-09-24'),
        pd.Timestamp('2017-10-09'),
        pd.Timestamp('2017-10-22'),
        pd.Timestamp('2017-11-12'),
        pd.Timestamp('2017-11-19'),
        pd.Timestamp('2017-12-03'),
    },
    '2018': {
        pd.Timestamp('2018-09-17'),
        pd.Timestamp('2018-09-30'),
        pd.Timestamp('2018-10-21'),
        pd.Timestamp('2018-10-28'),
        pd.Timestamp('2018-11-11'),
        pd.Timestamp('2018-11-18'),
        pd.Timestamp('2018-12-09'),
        pd.Timestamp('2018-12-16'),
    }
}

# now need to separate based on game-day or non-game day entries
s_gd, s_notgd = split_df_on_game_days(soldiers_df, gds_by_yr)
s_gd["treated"] = 1;    s_gd["game_day"] = 1
s_notgd["treated"] = 1; s_notgd["game_day"] = 0

#same now for jackson parkl
j_gd, j_notgd = split_df_on_game_days(jackson_df, gds_by_yr)
j_gd["treated"] = 0;    j_gd["game_day"] = 1
j_notgd["treated"] = 0; j_notgd["game_day"] = 0

##trying wicker too now
w_gd, w_notgd = split_df_on_game_days(wicker_df, gds_by_yr)
w_gd["treated"] = 0;    w_gd["game_day"] = 1
w_notgd["treated"] = 0; w_notgd["game_day"] = 0

GAME DAY
       start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi
553997 2017-09-10                2  101.0  41.876511 -87.620548      0.988131
553998 2017-09-10                3  247.0  41.867226 -87.615355      0.333785
553999 2017-09-10                4  130.0  41.856268 -87.613348      0.463860
554000 2017-09-10                5   23.0  41.874053 -87.627716      0.979012
554034 2017-09-10               41   37.0  41.872078 -87.629544      0.935233
...           ...              ...    ...        ...        ...           ...
768771 2018-12-16              341   18.0  41.866095 -87.607267      0.545252
768786 2018-12-16              370    2.0  41.854184 -87.619154      0.588281
768796 2018-12-16              394    7.0  41.870816 -87.631246      0.943576
768894 2018-12-16              623   16.0  41.872773 -87.623981      0.802603
768897 2018-12-16              626   11.0  41.867491 -87.632190      0.868451

[313 rows x 6 columns]
*NOT* GAME DAY
       start_dat

In [150]:
#now with the 4 separate datasets (for A,B,C,D analagously in DiD table),
# can do the DiD calcualtion using th statsmodels api....

### JACKSON PARK DiD BEFORE FE ADDED ### 
combined = pd.concat ([s_gd, s_notgd, j_gd, j_notgd], ignore_index=True)
daily = (combined.groupby(["start_date", "treated", "game_day"])["trips"].sum().reset_index())
daily["DiD"] = daily["treated"] * daily["game_day"]
# print(daily)

#now can run OLS 
X = sm.add_constant(daily[["treated", "game_day", "DiD"]])
y = daily["trips"]

model = sm.OLS(y, X).fit() #model for SF=tmt, JP=ctrl; no FE added yet (--> DiD coeff =144, p=0.120 ...)
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  trips   R-squared:                       0.485
Model:                            OLS   Adj. R-squared:                  0.480
Method:                 Least Squares   F-statistic:                     109.1
Date:                Wed, 20 May 2026   Prob (F-statistic):           8.13e-50
Time:                        19:27:50   Log-Likelihood:                -2430.3
No. Observations:                 352   AIC:                             4869.
Df Residuals:                     348   BIC:                             4884.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         77.8944     19.110      4.076      0.0

In [151]:
### ***WICKER*** PARK DiD BEFORE FE ADDED ### 
combined = pd.concat ([s_gd, s_notgd, w_gd, w_notgd], ignore_index=True)
daily = (combined.groupby(["start_date", "treated", "game_day"])["trips"].sum().reset_index())

daily["DiD"] = daily["treated"] * daily["game_day"]

#now can run OLS
X = sm.add_constant(daily[["treated", "game_day", "DiD"]])
y = daily["trips"]

model = sm.OLS(y, X).fit() #model for SF=tmt, WP=ctrl; no FE added yet (--> DiD coeff =232, p=0.039 (woah hello?? ... p<0.05 even before FE) ...)
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  trips   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     2.021
Date:                Wed, 20 May 2026   Prob (F-statistic):              0.111
Time:                        19:27:50   Log-Likelihood:                -2497.7
No. Observations:                 352   AIC:                             5003.
Df Residuals:                     348   BIC:                             5019.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        510.6584     23.147     22.061      0.0

In [152]:
#### NOW ADDING: Station, Month, Holiday, DoW, Year FEs ####
### JACKSON PARK ###

# Rebuild combined at station-day level (NOT the aggregated daily df)
# s_gd, s_notgd, j_gd, j_notgd already have treated + game_day tags from prev cell
# just re-stack them directly
J_df = pd.concat([s_gd, s_notgd, j_gd, j_notgd], ignore_index=True)

#Add DiD interaction + time variables
J_df["DiD"]        = J_df["treated"] * J_df["game_day"]
J_df["start_date"] = pd.to_datetime(J_df["start_date"])
J_df["dow"]        = J_df["start_date"].dt.dayofweek   # 0=Mon, 6=Sun
J_df["month"]      = J_df["start_date"].dt.month        # 1–12 for seasonality
J_df["year"]       = J_df["start_date"].dt.year

#for holdiay FE
holidays = {
    pd.Timestamp('2017-11-23'),  # Thanksgiving 2017
    pd.Timestamp('2017-11-24'),  # Black Friday
    pd.Timestamp('2018-11-22'),  # Thanksgiving 2018
    pd.Timestamp('2018-11-23'),  # Black Friday
}
J_df["holiday"] = J_df["start_date"].isin(holidays).astype(int)

# Filter to 2017–2018 study window only
J_df = J_df[J_df["year"].isin([2017, 2018])].copy()
J_df["from_station_id"] = J_df["from_station_id"].astype(int)  # clean up for C() dummies

print(J_df)

print(f"Rows in study window: {len(J_df):,}")
print(f"Game-day rows:        {J_df['game_day'].sum():,}")
print(f"Unique stations:      {J_df['from_station_id'].nunique()}")

     start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi  \
0    2017-09-10                2  101.0  41.876511 -87.620548      0.988131   
1    2017-09-10                3  247.0  41.867226 -87.615355      0.333785   
2    2017-09-10                4  130.0  41.856268 -87.613348      0.463860   
3    2017-09-10                5   23.0  41.874053 -87.627716      0.979012   
4    2017-09-10               41   37.0  41.872078 -87.629544      0.935233   
...         ...              ...    ...        ...        ...           ...   
5040 2018-12-15              345    3.0  41.793242 -87.587782      0.763465   
5041 2018-12-15              352    1.0  41.773518 -87.577143      0.706007   
5042 2018-12-15              424    3.0  41.791728 -87.583945      0.605396   
5043 2018-12-15              425    3.0  41.787943 -87.588315      0.470317   
5044 2018-12-15              426    2.0  41.785097 -87.601073      0.997387   

      treated  game_day  DiD  dow  month  year  hol

In [153]:
### NOW ADD VARS FOR TIME, STATION FEs, BUT FOR ***WICKER*** ###

# Rebuild combined at station-day level (NOT the aggregated daily df)
# s_gd, s_notgd, w_gd, w_notgd already have treated + game_day tags from prev cell
# just re-stack them directly
W_df = pd.concat([s_gd, s_notgd, w_gd, w_notgd], ignore_index=True)

#Add DiD interaction + time variables
W_df["DiD"]        = W_df["treated"] * W_df["game_day"]
W_df["start_date"] = pd.to_datetime(W_df["start_date"])
W_df["dow"]        = W_df["start_date"].dt.dayofweek   # 0=Mon, 6=Sun
W_df["month"]      = W_df["start_date"].dt.month        # 1–12 for seasonality
W_df["year"]       = W_df["start_date"].dt.year

#for holdiay FE
holidays = {
    pd.Timestamp('2017-11-23'),  # Thanksgiving 2017
    pd.Timestamp('2017-11-24'),  # Black Friday
    pd.Timestamp('2018-11-22'),  # Thanksgiving 2018
    pd.Timestamp('2018-11-23'),  # Black Friday
}
W_df["holiday"] = W_df["start_date"].isin(holidays).astype(int)

# Filter to 2017–2018 study window only
W_df = W_df[W_df["year"].isin([2017, 2018])].copy()
W_df["from_station_id"] = W_df["from_station_id"].astype(int)  # clean up for C() dummies

print(W_df)

print(f"Rows in study window: {len(W_df):,}")
print(f"Game-day rows:        {W_df['game_day'].sum():,}")
print(f"Unique stations:      {W_df['from_station_id'].nunique()}")

     start_date  from_station_id  trips   Latitude  Longitude  dist_to_c_mi  \
0    2017-09-10                2  101.0  41.876511 -87.620548      0.988131   
1    2017-09-10                3  247.0  41.867226 -87.615355      0.333785   
2    2017-09-10                4  130.0  41.856268 -87.613348      0.463860   
3    2017-09-10                5   23.0  41.874053 -87.627716      0.979012   
4    2017-09-10               41   37.0  41.872078 -87.629544      0.935233   
...         ...              ...    ...        ...        ...           ...   
8248 2018-12-15              374    9.0  41.898418 -87.686596      0.819203   
8249 2018-12-15              628    2.0  41.914610 -87.667968      0.655305   
8250 2018-12-15              637    3.0  41.895634 -87.672069      0.870812   
8251 2018-12-15              657    5.0  41.899181 -87.672200      0.636429   
8252 2018-12-15              659    8.0  41.895501 -87.682017      0.888069   

      treated  game_day  DiD  dow  month  year  hol

In [162]:
def run_triple_spec(df:pd.DataFrame): #made into function to stop doubling everything LOL too lazy to go back and do same for above oops

    #Spec 1: bare DiD (matches existing basic model as sanity check )
    m1 = smf.ols(
        'trips ~ treated + game_day + DiD',
        data=df
    ).fit(cov_type='HC3')

    #Spec 2: + day-of-week and month fixed effects
    m2 = smf.ols(
        'trips ~ treated + game_day + DiD + C(dow) + C(month)',
        data=df
    ).fit(cov_type='HC3')

    #Spec 3: + station fixed effects (strongest
    # 'treated' drops out here — it's constant within each station, absorbed by station FE
    m3 = smf.ols(
        'trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month)',
        data=df
    ).fit(cov_type='HC3')

    #Print clean comparison table
    table = summary_col(
        [m1, m2, m3],
        stars=True,
        model_names=['(1) Basic DiD', '(2) +DOW/Month FE', '(3) +Station FE'],
        info_dict={
            'N':  lambda x: f"{int(x.nobs):,}",
            'R²': lambda x: f"{x.rsquared:.3f}"
        },
        regressor_order=['DiD', 'treated', 'game_day'],
        drop_omitted=True
    )
    print(table)


    #Also print j the DiD row cleanly 
    print("\n=== DiD estimate across specs ===")
    for name, m in [('(1)', m1), ('(2)', m2), ('(3)', m3)]:
        print(f"{name}  β={m.params['DiD']:8.3f}  SE={m.bse['DiD']:7.3f}  p={m.pvalues['DiD']:.3f}")

In [163]:
print("wicker PARK as ctrl:")
run_triple_spec(W_df)
print("\n-------------------------------\n")
print("JACKSON PARK as ctrl:")
run_triple_spec(J_df)

wicker PARK as ctrl:

               (1) Basic DiD (2) +DOW/Month FE (3) +Station FE
--------------------------------------------------------------
DiD            10.3187***    10.2500***        10.0157***     
               (2.4691)      (2.2356)          (1.8102)       
treated        5.7231***     5.4023***                        
               (0.5266)      (0.4730)                         
game_day       -3.5228***    -2.2478*          -2.0264**      
               (0.8594)      (1.1710)          (0.9180)       
R-squared      0.0251        0.2108            0.5426         
R-squared Adj. 0.0247        0.2096            0.5392         
N              8,253         8,253             8,253          
R²             0.025         0.211             0.543          
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01

=== DiD estimate across specs ===
(1)  β=  10.319  SE=  2.469  p=0.000
(2)  β=  10.250  SE=  2.236  p=0.000
(3)  β=  10.016  SE=  1.810  p=0.000

----------------

In [164]:
### ADDING: WEATHER [ 1) temp, 2) wind speed] FEs ###

def get_weather(lat, lon, start, end):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "daily": ["temperature_2m_max", "windspeed_10m_max"],
        "timezone": "America/Chicago",
        "temperature_unit": "fahrenheit",
        "windspeed_unit": "mph"
    }
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)
    d = r.json()["daily"]
    return pd.DataFrame({
        "start_date": pd.to_datetime(d["time"]),
        "temp_max":   d["temperature_2m_max"],
        "wind_max":   d["windspeed_10m_max"]
    })

soldiers_weather = get_weather(soldiers_coords[0], soldiers_coords[1], "2017-09-10", "2018-12-16")
soldiers_weather["treated"] = 1
print(soldiers_weather) #just chekcing looks right 

### JACKSON ###
jackson_weather  = get_weather(jackson_coords[0], jackson_coords[1], "2017-09-10", "2018-12-16")
jackson_weather["treated"] = 0

### WICKER ###
wicker_weather  = get_weather(wicker_coords[0], wicker_coords[1], "2017-09-10", "2018-12-16")
wicker_weather["treated"] = 0

SJ_weather = pd.concat([soldiers_weather, jackson_weather], ignore_index=True)
SW_weather = pd.concat([soldiers_weather, wicker_weather], ignore_index=True)


    start_date  temp_max  wind_max  treated
0   2017-09-10      67.0      10.1        1
1   2017-09-11      68.6       9.3        1
2   2017-09-12      71.4      12.9        1
3   2017-09-13      71.3       8.4        1
4   2017-09-14      78.5      10.9        1
..         ...       ...       ...      ...
458 2018-12-12      41.7      19.2        1
459 2018-12-13      43.1       9.7        1
460 2018-12-14      40.5      12.5        1
461 2018-12-15      40.9      13.3        1
462 2018-12-16      48.0      11.2        1

[463 rows x 4 columns]


In [166]:
### WICKER: merge into sw_df, add wind adv flag

SW_df = W_df.merge(SW_weather, on=["start_date", "treated"], how="left")

# Wind advisory flag: NWS threshold is sustained 25+ mph
SW_df["wind_advisory"] = (SW_df["wind_max"] >= 25).astype(int)

### JACKSON: ibid
SJ_df = J_df.merge(SJ_weather, on=["start_date", "treated"], how="left")

# Wind advisory flag: NWS threshold is sustained 25+ mph
SJ_df["wind_advisory"] = (SJ_df["wind_max"] >= 25).astype(int)

In [169]:
# add to regress ion specs
# m2 = smf.ols(
#     'trips ~ treated + game_day + DiD + C(dow) + C(month) + temp_max + wind_advisory',
#     data=df
# ).fit(cov_type='HC3')

def update_specs_all_FEs(df: pd.DataFrame):
    #Spec 1: bare DiD (matches existing basic model as sanity check )
    m1 = smf.ols(
        'trips ~ treated + game_day + DiD',
        data=df
    ).fit(cov_type='HC3')

    #new m2 after adding year and holdiay FEs (on top of month, DoW, weather [<--temp, wind])
    m2 = smf.ols(
        'trips ~ treated + game_day + DiD + C(dow) + C(month) + C(year) + holiday + temp_max + wind_advisory',
        data=df
    ).fit(cov_type='HC3')

    # m3 = smf.ols(
    #     'trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month) + temp_max + wind_advisory',
    #     data=df
    # ).fit(cov_type='HC3')

    #new m3 after adding year and holdiay FEs (on top of all prev in m2 PLUS added Station FE)
    m3 = smf.ols(
        'trips ~ game_day + DiD + C(from_station_id) + C(dow) + C(month) + C(year) + holiday + temp_max + wind_advisory',
        data=df
    ).fit(cov_type='HC3')
    return m1, m2, m3

w_m1, w_m2, w_m3 = update_specs_all_FEs(SW_df)
j_m1, j_m2, j_m3 = update_specs_all_FEs(SJ_df)


In [170]:
# regression now w ALL FEs incl. fixed effecta for weather, REORGANIZED SPECS 1) bare DiD , 2) DiD w/ Month/year/DoW/weather/holiday FEs, 3) all in (2) plus station FEs

def run_trip_reg_all_FEs(m1, m2, m3):
    table = summary_col(
        [m1, m2, m3],
        stars=True,
        model_names=['(1) Basic DiD', '(2) +DOW/Month/Year/Holiday/Weather', '(3) +Station FE'],
        info_dict={
            'N':  lambda x: f"{int(x.nobs):,}",
            'R²': lambda x: f"{x.rsquared:.3f}"
        },
        regressor_order=['DiD', 'treated', 'game_day', 'temp_max', 'wind_advisory', 'holiday', 'C(year)', 'C(month)', 'C(dow)', 'C(from_station_id)'],
        drop_omitted=True
    )
    print(table)

    print("\n=== DiD estimate across specs ===")
    for name, m in [('(1)', m1), ('(2)', m2), ('(3)', m3)]:
        print(f"{name}  β={m.params['DiD']:8.3f}  SE={m.bse['DiD']:7.3f}  p={m.pvalues['DiD']:.3f}")



In [171]:
### WICKER triple reg all FEs
run_trip_reg_all_FEs(w_m1, w_m2, w_m3)



               (1) Basic DiD (2) +DOW/Month/Year/Holiday/Weather (3) +Station FE
--------------------------------------------------------------------------------
DiD            10.3187***    10.2489***                          10.0089***     
               (2.4691)      (2.1867)                            (1.7562)       
treated        5.7231***     5.0348***                                          
               (0.5266)      (0.4557)                                           
game_day       -3.5228***    -1.7235                             -1.5557*       
               (0.8594)      (1.1358)                            (0.8803)       
temp_max                     0.3970***                           0.4064***      
                             (0.0273)                            (0.0201)       
wind_advisory                -5.4776***                          -5.8565***     
                             (0.8148)                            (0.7315)       
holiday                    

In [172]:
### JACKSON triple reg all FEs
run_trip_reg_all_FEs(j_m1, j_m2, j_m3)


               (1) Basic DiD (2) +DOW/Month/Year/Holiday/Weather (3) +Station FE
--------------------------------------------------------------------------------
DiD            6.8684***     7.9934***                           8.1404***      
               (2.5843)      (2.4896)                            (1.9979)       
treated        15.3710***    17.0542***                                         
               (0.5561)      (0.5949)                                           
game_day       -0.0725       -0.8404                             -0.8361        
               (1.1492)      (1.8203)                            (1.4562)       
temp_max                     0.3803***                           0.4105***      
                             (0.0373)                            (0.0285)       
wind_advisory                -6.1716***                          -7.2385***     
                             (1.4377)                            (1.3312)       
holiday                    

In [174]:
###### TRIPPLE DiD (DDD) #########
# will use Jackson/SF DiD and Wicker/SF DiD to construct
# Dimensions:
# - Location: near Soldier Field (treated=1) vs. far (treated=0)
# - Time: game day (game_day=1) vs. non-game day (game_day=0)
# - Stratum: lakefront/urban (urban=1) vs. inland park (urban=0)

#the 6 datasets alr exist w/ treated + game_day tags
#just need to add which comparison ea. row belongs to

#Soldier Field rows appear in both comparisons.. duplicate:
s_gd_w = s_gd.copy();     s_gd_w["wicker_comparison"]    = 1
s_notgd_w = s_notgd.copy(); s_notgd_w["wicker_comparison"] = 1
s_gd_j = s_gd.copy();     s_gd_j["wicker_comparison"]    = 0
s_notgd_j = s_notgd.copy(); s_notgd_j["wicker_comparison"] = 0

w_gd["wicker_comparison"]    = 1
w_notgd["wicker_comparison"] = 1
j_gd["wicker_comparison"]    = 0
j_notgd["wicker_comparison"] = 0

df_ddd = pd.concat([
    s_gd_w, s_notgd_w,    # SF in Wicker comparison
    s_gd_j, s_notgd_j,    # SF in Jackson comparison
    w_gd,   w_notgd,       # Wicker control
    j_gd,   j_notgd,       # Jackson control
], ignore_index=True)

df_ddd["DiD"] = df_ddd["treated"] * df_ddd["game_day"]
df_ddd["DDD"] = df_ddd["treated"] * df_ddd["game_day"] * df_ddd["wicker_comparison"]

df_ddd["start_date"] = pd.to_datetime(df_ddd["start_date"])
df_ddd["dow"]   = df_ddd["start_date"].dt.dayofweek
df_ddd["month"] = df_ddd["start_date"].dt.month
#adding year and holiday FEs
df_ddd["year"]    = df_ddd["start_date"].dt.year
df_ddd["holiday"] = df_ddd["start_date"].isin(holidays).astype(int)
df_ddd = df_ddd[df_ddd["start_date"].dt.year.isin([2017, 2018])].copy()
df_ddd["from_station_id"] = df_ddd["from_station_id"].astype(int)

# Build a weather df keyed by (start_date, treated, wicker_comparison)
soldiers_w = soldiers_weather.copy(); soldiers_w["wicker_comparison"] = 1
soldiers_j = soldiers_weather.copy(); soldiers_j["wicker_comparison"] = 0
wicker_w   = wicker_weather.copy();   wicker_w["treated"] = 0; wicker_w["wicker_comparison"] = 1
jackson_j  = jackson_weather.copy();  jackson_j["treated"] = 0; jackson_j["wicker_comparison"] = 0

ddd_weather = pd.concat([soldiers_w, soldiers_j, wicker_w, jackson_j], ignore_index=True)

# Merge into df_ddd
df_ddd = df_ddd.drop(columns=["temp_max", "wind_max", "wind_advisory"], errors="ignore")
df_ddd = df_ddd.merge(ddd_weather, on=["start_date", "treated", "wicker_comparison"], how="left")
df_ddd["wind_advisory"] = (df_ddd["wind_max"] >= 25).astype(int)

# m_ddd = smf.ols(
#     """trips ~ treated + game_day + wicker_comparison
#              + DiD
#              + treated:wicker_comparison
#              + game_day:wicker_comparison
#              + DDD
#              + C(from_station_id) + C(dow) + C(month)""",
#     data=df_ddd
# ).fit(cov_type='HC3')

# #updated m_ddd after adding year and holdiay FEs
# m_ddd = smf.ols(
#     """trips ~ treated + game_day + wicker_comparison
#              + DiD
#              + treated:wicker_comparison
#              + game_day:wicker_comparison
#              + DDD
#              + C(dow) + C(month) + C(year) + holiday""",
#     data=df_ddd
# ).fit(cov_type='HC3')

# updated m_ddd including weather along all other FEs (Except station FE)
m_ddd_1 = smf.ols(
    """trips ~ treated + game_day + wicker_comparison
             + DiD
             + treated:wicker_comparison
             + game_day:wicker_comparison
             + DDD
             + C(dow) + C(month) + C(year) + holiday
             + temp_max + wind_advisory""",
    data=df_ddd
).fit(cov_type='HC3')


#Full summary w all coefficients
print(m_ddd_1.summary())

#Clean table showing only the terms we care about
#(hides the station/dow/month FE dummies)
table = summary_col(
    [m_ddd_1],
    stars=True,
    model_names=['DDD'],
    info_dict={
        'N':  lambda x: f"{int(x.nobs):,}",
        'R²': lambda x: f"{x.rsquared:.3f}"
    },
    regressor_order=['DiD', 'DDD', 'treated', 'game_day', 'wicker_comparison'],
    drop_omitted=True
)
print(table)

# j the key rows printed cleanly
print("\n=== Key DDD estimates ===")
for name, label in [
    ('DiD', 'DiD β (Jackson ctrl)   '),
    ('DDD', 'DDD β (Wicker−Jackson) '),
]:
    print(f"{label}  β={m_ddd_1.params[name]:8.3f}  "
          f"SE={m_ddd_1.bse[name]:6.3f}  "
          f"p={m_ddd_1.pvalues[name]:.4f}")

print(f"\nImplied Wicker DiD β:   "
      f"{m_ddd_1.params['DiD'] + m_ddd_1.params['DDD']:.3f}")

print(f"DiD β (Jackson as control):        {m_ddd_1.params['DiD']:.3f}")
print(f"DDD β (Wicker vs Jackson control): {m_ddd_1.params['DDD']:.3f}")
print(f"DDD p-value:                       {m_ddd_1.pvalues['DDD']:.4f}")
print(f"\nImplied DiD with Wicker: {m_ddd_1.params['DiD'] + m_ddd_1.params['DDD']:.3f}")




                            OLS Regression Results                            
Dep. Variable:                  trips   R-squared:                       0.247
Model:                            OLS   Adj. R-squared:                  0.246
Method:                 Least Squares   F-statistic:                     181.4
Date:                Wed, 20 May 2026   Prob (F-statistic):               0.00
Time:                        19:59:44   Log-Likelihood:                -59177.
No. Observations:               13298   AIC:                         1.184e+05
Df Residuals:                   13277   BIC:                         1.186e+05
Df Model:                          20                                         
Covariance Type:                  HC3                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           